# Plotting and Results Comparison

## Imports

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

import io
import base64
import json

## Configuration

In [ ]:
RESULTS_DIR = Path("../../results/")
TRUE_LABEL_COLUMN = 'y'
PRED_SCORE_COLUMN = 'pred'

### Sign into a Rhino Session
Provide the email associated with your account in the `USERNAME` variable, and when prompted, provide your password.

In [ ]:
from getpass import getpass
import rhino_health as rh

USERNAME = 'username' # Change this to your Rhino FCP Username

print("Enter Your Password:")

session = rh.login(username=USERNAME, password=getpass())

print("Logged In")

## Plotting ROC Curves
Here we have defined a function to plot an ROC curve when given a dictionary, where each entry is another dictionary entry containing a datasets FPR, TPR, and AUC values.

In [ ]:
def plot_roc_curve(roc_data: dict, title: str = "ROC Curve Comparison"):
    sorted_roc_data = dict(sorted(roc_data.items(), key=lambda x: x[1][2], reverse=True))
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)

    for name, (fpr, tpr, roc_auc) in sorted_roc_data.items():
        ax.plot(
            fpr, tpr,
            label=f'{name} (AUC = {roc_auc:.4f})',
            linewidth=2
        )

    
    ax.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.5000)')

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(alpha=0.4)
    
    plt.close(fig)

    return fig

## Plotting Local Results
We start by reading in each inference dataset from local/simulated training inferences, obtaining their FPR, TPR, and AUC values, and then passing the results into our plotting function.

In [ ]:
metric_results = {}

# Iterate over each dataset in the results directory
for dataset in RESULTS_DIR.iterdir():
    # Extract the site name from the dataset filename
    site_name = dataset.stem.split("local_")[-1].split("_preds")[0].replace("_", " ").title()
    
    # Read the dataset into a pandas DataFrame
    df = pd.read_csv(dataset)
    
    # Extract the true labels and predicted scores
    y_true = df[TRUE_LABEL_COLUMN].values
    y_scores = df[PRED_SCORE_COLUMN].values
    
    # Calculate the TPR, FPR, and AUC values
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    # Store the results
    metric_results[f'{site_name} Local Model'] = (fpr, tpr, roc_auc)

display(plot_roc_curve(metric_results, title="ROC Curve for Model Performance"))

Analyzing the results, we see the simulated FL training and centralized data training both outperformed the best performing individual site. This highlights an important aspect of collaborative training. There may not always be equitable gains across sites when compared to a global model, but when federated learning is conducted appropriately, all collaborators should see **some** gain.

## SDK Plotting

### Selecting our Dataset
Now we should compare our results against the training done on the Rhino FCP.

First we must select our dataset by providing the dataset's UID to the `fl_dataset_uid` variable.

This can be obtained by doing the following:
* Navigate to the Datasets tab within the Rhino FCP project
* Navigate to the most recent version of the dataset created by the validation code run
* Click on the **&#8942;** icon on the right side of the dataset of interest
* Select Copy UID

In [ ]:
# Povide dataset UID here
fl_dataset_uid = "fl_dataset_uid"

#This is where the session creates the reference to the dataset of interest
fl_dataset = session.dataset.get_dataset(fl_dataset_uid)

print(f"Loaded dataset '{fl_dataset.name}'")

### Configuring our Metric

The Rhino SDK contains a broad suite of statistical tools to enable decentralized data analysis. See [Calculating Federated Analytics](https://docs.rhinohealth.com/hc/en-us/articles/12385294342173-Calculating-Federated-Analytics) for more information on the different functions available and how to execute them.

Here we have set up a configuration to calculate the ROCAUC metrics needed to create our plots.

In [ ]:
# Import the Rhino SDK RocAuc metric
from rhino_health.lib.metrics import RocAuc

# Define our metric configuration
metric_configuration = RocAuc(y_true_variable="y", y_pred_variable="pred")

# Calculate the TPR, FPR, and AUC values
fl_results = fl_dataset.get_metric(metric_configuration)

# Extract the FPR, TPR, and AUC values
fpr = np.array(fl_results.output["fpr"])
tpr = np.array(fl_results.output["tpr"])
roc_auc = fl_results.output["auc"]

# Store the results
metric_results["Federated Model"] = (fpr, tpr, roc_auc)

# Plot the results
fig = plot_roc_curve(metric_results, title="ROC Curve for Model Performance")
display(fig)

Reanalyzing the results, we see the model trained within the Rhino FCP performs nearly identical. It likely does perform identically since we used deterministic training, but differences may present themselves due to security features like [Differential Privacy](https://docs.rhinohealth.com/hc/en-us/articles/14227409711133-Differential-Privacy) injecting small amounts of noise into the data.

### Post the results
Below we have provided a function which allows the user to post plots to the Rhino FCP and connect them to a code object.

In [ ]:
def post_results(figs, code_object_uid):
    report_data = [{"type": "Title", "data": "Experiment Results"}]
    for fig in figs:
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200)
        buf.seek(0)
        base_64_image = base64.b64encode(buf.getvalue()).decode("utf-8")

        report_data.append({
            "type": "Image",
            "data": {
                "image_filename": "experiment_ROCs.png",
                "image_base64": base_64_image
            },
                 "width": 100 / len(figs)
        })

    _ = session.post(f"code_runs/{code_object_uid}/set_report/",
                data={"report_data": json.dumps(report_data)})

In this case, we'd like to post the plot we just generated to the inference code run associated with our inference dataset.

In the cell below, provide the inference code run's UID to `code_object_uid`. This can be obtained in a similar way to the dataset UID but accesible through the Code Runs tab within the project.
* Navigate to the Code Runs tab within the Rhino FCP project
* Navigate to the Validation (**V**) Code Run
* Click on the **&#8942;** icon on the right side of that Code Run
* Select Copy UID

In [ ]:
# Define the code run UID
code_run_uid = "code_run_uid"

post_results([fig], code_run_uid)

You can now navigate back to your project and view the attached report alongside the Validation Code Run!